# conv-channel-sum composite — cx6: 1x1 conv as channel-sum via einsum 'bchw,ochw->bohw'

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-channel-sum`, `einops-einsum`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-channel-sum"
DD_ATOM_IDS = ["conv-channel-sum", "einops-einsum"]
DD_SUBTOPICS = ["CNN: Channel-axis sum semantics", "Einops: Deep Learning"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A `1x1` convolution is the pure channel-sum atom with no spatial windowing:
  `out[b, o, h, w] = sum_c x[b, c, h, w] * w[o, c, h, w]`  (spatially aligned).

More commonly the 1x1 conv is written with a `(C_out, C_in, 1, 1)` weight — but if we imagine the spatially-broadcast version where `w` has shape `(C_out, C_in, H, W)`, the operation becomes a clean einsum:

  `einops.einsum(x, w, 'b c h w, o c h w -> b o h w')`

Two atoms:
- **conv-channel-sum** — the `c` label appears on BOTH inputs but not on the output →   einsum contracts over c, summing the C_in axis.
- **einops-einsum** — `h, w` appear on both inputs AND the output → preserved (no   reduction along spatial axes; this is the channel-only flavor of a conv).

The composition: write the einsum, and verify it matches a hand-rolled channel-sum reduction over the same shapes.

### Composite Exercise — 1x1 conv as channel-sum via einsum 'bchw,ochw->bohw'

**Atoms exercised together**: `conv-channel-sum`, `einops-einsum`

Implement `cx6_channel_sum_einsum(x, w)`.

- `x`: float tensor of shape `(B, C_in, H, W)`.
- `w`: float tensor of shape `(C_out, C_in, H, W)` — note: SAME spatial dims as `x`. This is the spatially-broadcast form of a per-pixel channel mix.

Return a float tensor of shape `(B, C_out, H, W)` such that
  `out[b, o, h, w] = sum_c x[b, c, h, w] * w[o, c, h, w]`.

Use ONE `einops.einsum` call. The string MUST contract over `c` and preserve `b, o, h, w`.

1. **einops-einsum atom** — string of the form `'b c h w, o c h w -> b o h w'`.
2. **conv-channel-sum atom** — the only contracted axis is `c` (C_in), which is what makes this the channel-sum reduction.

The test cross-checks against the explicit `.sum(dim=2)` form and against a 1x1-conv reference.

In [ ]:
def cx6_channel_sum_einsum(x, w):
    # Atoms: einops-einsum string with `c` shared on both inputs but absent from output
    # = einsum contracts over c = the conv-channel-sum reduction.
    return einops.einsum(x, w, 'b c h w, o c h w -> b o h w')


<details><summary>Show solution — cx6</summary>

```python
def cx6_channel_sum_einsum(x, w):
    # Atoms: einops-einsum string with `c` shared on both inputs but absent from output
    # = einsum contracts over c = the conv-channel-sum reduction.
    return einops.einsum(x, w, 'b c h w, o c h w -> b o h w')
```

Reading the einsum: `c` is shared but missing from the right → contraction over C_in. Every other label (`b, o, h, w`) is preserved unchanged. This is the cleanest possible channel-sum reduction — no spatial windowing, no kernel. ARENA's full `conv2d_minimal` differs only by adding `kh kw` labels to both inputs (so they contract too).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx6',
        'subtopics': ["CNN: Channel-axis sum semantics", "Einops: Deep Learning"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()